In [1]:
########## Paketler ############
import asyncio
import aiohttp
import aiofiles
import pandas as pd
from bs4 import BeautifulSoup
import time
import random
from pathlib import Path
import nest_asyncio



In [2]:
########## Haberler ############

# Asenkron istekler için gerekli
nest_asyncio.apply()

async def fetch_page(session, url, semaphore):
    """Tek bir sayfayı asenkron olarak getir"""
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as response:
                if response.status == 200:
                    html = await response.text()
                    return html
                else:
                    print(f"HTTP {response.status} for {url}")
                    return None
        except Exception as e:
            print(f"Error fetching {url}: {str(e)}")
            return None

def parse_haber_links(html, page_num):
    """HTML'den haber linklerini parse et"""
    if not html:
        return []
    
    soup = BeautifulSoup(html, 'html.parser')
    page_links = [a['href'] for a in soup.find_all('a', href=True) if a.get('href')]
    
    haber_links = []
    for link in page_links:
        if '/haberler/s/' in link:
            haber_links.append(link)
    
    print(f"Page {page_num}: Found {len(haber_links)} links", end='\r')
    return haber_links

async def process_single_page(session, page_num, semaphore):
    """Tek bir sayfayı işle"""
    if page_num == 1:
        url = "https://www.tepav.org.tr/tr/haberler"
    else:
        url = f"https://www.tepav.org.tr/tr/haberler?page={page_num}"
    
    html = await fetch_page(session, url, semaphore)
    if html:
        links = parse_haber_links(html, page_num)
        return links
    return []

async def main():
    """Ana işlem fonksiyonu"""
    start_time = time.time()
    
    # Eşzamanlı istek sayısını sınırla (sitelere saygı için)
    semaphore = asyncio.Semaphore(10)
    
    # Tüm sayfaları asenkron olarak işle
    async with aiohttp.ClientSession() as session:
        tasks = []
        for page_num in range(1, 1000):
            task = process_single_page(session, page_num, semaphore)
            tasks.append(task)
        
        # Tüm görevleri paralel çalıştır
        results = await asyncio.gather(*tasks)
    
    # Tüm linkleri birleştir
    all_links = []
    for links in results:
        all_links.extend(links)
    
    # Benzersiz linkler
    unique_links = list(set(all_links))
    
    # DataFrame oluştur ve kaydet
    df = pd.DataFrame({
        'link': ['/tr' + z.replace('//', '/') for z in unique_links]
    })
    
    df.to_excel('haberler.xlsx', index=False)
    
    end_time = time.time()
    print(f"\nToplam süre: {end_time - start_time:.2f} saniye")
    print(f"Toplam {len(unique_links)} benzersiz haber linki bulundu")

# Batch processing için alternatif versiyon
async def process_batch(session, page_range, semaphore):
    """Sayfa gruplarını işle"""
    tasks = []
    for page_num in page_range:
        task = process_single_page(session, page_num, semaphore)
        tasks.append(task)
    
    batch_results = await asyncio.gather(*tasks)
    return batch_results

async def main_batched(batch_size=50):
    """Batch processing ile daha kontrollü versiyon"""
    start_time = time.time()
    
    semaphore = asyncio.Semaphore(15)  # Daha fazla eşzamanlı istek
    all_links = []
    
    async with aiohttp.ClientSession() as session:
        for batch_start in range(1, 1000, batch_size):
            batch_end = min(batch_start + batch_size, 1000)
            batch_range = range(batch_start, batch_end)
            
            print(f"Processing batch {batch_start}-{batch_end-1}")
            
            batch_results = await process_batch(session, batch_range, semaphore)
            
            for links in batch_results:
                all_links.extend(links)
            
            # Küçük bir bekleme süresi
            await asyncio.sleep(1)
    
    # Benzersiz linkler
    unique_links = list(set(all_links))
    
    # Kaydet
    df = pd.DataFrame({
        'link': ['/tr' + z.replace('//', '/') for z in unique_links]
    })
    
    df.to_excel('haberler.xlsx', index=False)
    
    end_time = time.time()
    print(f"\nToplam süre: {end_time - start_time:.2f} saniye")
    print(f"Toplam {len(unique_links)} benzersiz haber linki bulundu")

# Çalıştırma seçenekleri
if __name__ == "__main__":
    # Hızlı versiyon
    print("Hızlı versiyon çalışıyor...")
    asyncio.run(main())
    
    # Veya batch versiyon
    # print("Batch versiyon çalışıyor...")
    # asyncio.run(main_batched())

Hızlı versiyon çalışıyor...
Page 994: Found 5 linkss
Toplam süre: 29.20 saniye
Toplam 2735 benzersiz haber linki bulundu


In [3]:
########## Blog ############

# Asenkron istekler için gerekli
nest_asyncio.apply()

async def fetch_page(session, url, semaphore):
    """Tek bir sayfayı asenkron olarak getir"""
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as response:
                if response.status == 200:
                    html = await response.text()
                    return html
                else:
                    print(f"HTTP {response.status} for {url}")
                    return None
        except Exception as e:
            print(f"Error fetching {url}: {str(e)}")
            return None

def parse_haber_links(html, page_num):
    """HTML'den haber linklerini parse et"""
    if not html:
        return []
    
    soup = BeautifulSoup(html, 'html.parser')
    page_links = [a['href'] for a in soup.find_all('a', href=True) if a.get('href')]
    
    haber_links = []
    for link in page_links:
        if '/blog/s/' in link:
            haber_links.append(link)
    
    print(f"Page {page_num}: Found {len(haber_links)} links", end='\r')
    return haber_links

async def process_single_page(session, page_num, semaphore):
    """Tek bir sayfayı işle"""
    if page_num == 1:
        url = "https://www.tepav.org.tr/tr/blog"
    else:
        url = f"https://www.tepav.org.tr/tr/blog?page={page_num}"
    
    html = await fetch_page(session, url, semaphore)
    if html:
        links = parse_haber_links(html, page_num)
        return links
    return []

async def main():
    """Ana işlem fonksiyonu"""
    start_time = time.time()
    
    # Eşzamanlı istek sayısını sınırla (sitelere saygı için)
    semaphore = asyncio.Semaphore(10)
    
    # Tüm sayfaları asenkron olarak işle
    async with aiohttp.ClientSession() as session:
        tasks = []
        for page_num in range(1, 1000):
            task = process_single_page(session, page_num, semaphore)
            tasks.append(task)
        
        # Tüm görevleri paralel çalıştır
        results = await asyncio.gather(*tasks)
    
    # Tüm linkleri birleştir
    all_links = []
    for links in results:
        all_links.extend(links)
    
    # Benzersiz linkler
    unique_links = list(set(all_links))
    
    # DataFrame oluştur ve kaydet
    df = pd.DataFrame({
        'link': ['/tr' + z.replace('//', '/') for z in unique_links]
    })
    
    df.to_excel('blog.xlsx', index=False)
    
    end_time = time.time()
    print(f"\nToplam süre: {end_time - start_time:.2f} saniye")
    print(f"Toplam {len(unique_links)} benzersiz haber linki bulundu")

# Batch processing için alternatif versiyon
async def process_batch(session, page_range, semaphore):
    """Sayfa gruplarını işle"""
    tasks = []
    for page_num in page_range:
        task = process_single_page(session, page_num, semaphore)
        tasks.append(task)
    
    batch_results = await asyncio.gather(*tasks)
    return batch_results

async def main_batched(batch_size=50):
    """Batch processing ile daha kontrollü versiyon"""
    start_time = time.time()
    
    semaphore = asyncio.Semaphore(15)  # Daha fazla eşzamanlı istek
    all_links = []
    
    async with aiohttp.ClientSession() as session:
        for batch_start in range(1, 1000, batch_size):
            batch_end = min(batch_start + batch_size, 1000)
            batch_range = range(batch_start, batch_end)
            
            print(f"Processing batch {batch_start}-{batch_end-1}")
            
            batch_results = await process_batch(session, batch_range, semaphore)
            
            for links in batch_results:
                all_links.extend(links)
            
            # Küçük bir bekleme süresi
            await asyncio.sleep(1)
    
    # Benzersiz linkler
    unique_links = list(set(all_links))
    
    # Kaydet
    df = pd.DataFrame({
        'link': ['/tr' + z.replace('//', '/') for z in unique_links]
    })
    
    df.to_excel('blog.xlsx', index=False)
    
    end_time = time.time()
    print(f"\nToplam süre: {end_time - start_time:.2f} saniye")
    print(f"Toplam {len(unique_links)} benzersiz blog linki bulundu")

# Çalıştırma seçenekleri
if __name__ == "__main__":
    # Hızlı versiyon
    print("Hızlı versiyon çalışıyor...")
    asyncio.run(main())
    
    # Veya batch versiyon
    # print("Batch versiyon çalışıyor...")
    # asyncio.run(main_batched())

Hızlı versiyon çalışıyor...
Page 998: Found 5 linkss
Toplam süre: 38.80 saniye
Toplam 4516 benzersiz haber linki bulundu


In [ ]:
########## Yayın ############

# Asenkron istekler için gerekli
nest_asyncio.apply()

async def fetch_page(session, url, semaphore):
    """Tek bir sayfayı asenkron olarak getir"""
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as response:
                if response.status == 200:
                    html = await response.text()
                    return html
                else:
                    print(f"HTTP {response.status} for {url}")
                    return None
        except Exception as e:
            print(f"Error fetching {url}: {str(e)}")
            return None

def parse_haber_links(html, page_num):
    """HTML'den haber linklerini parse et"""
    if not html:
        return []
    
    soup = BeautifulSoup(html, 'html.parser')
    page_links = [a['href'] for a in soup.find_all('a', href=True) if a.get('href')]
    
    haber_links = []
    for link in page_links:
        if '/yayin/s/' in link:
            haber_links.append(link)
    
    print(f"Page {page_num}: Found {len(haber_links)} links", end='\r')
    return haber_links

async def process_single_page(session, page_num, semaphore):
    """Tek bir sayfayı işle"""
    if page_num == 1:
        url = "https://www.tepav.org.tr/tr/yayin"
    else:
        url = f"https://www.tepav.org.tr/tr/yayin?page={page_num}"
    
    html = await fetch_page(session, url, semaphore)
    if html:
        links = parse_haber_links(html, page_num)
        return links
    return []

async def main():
    """Ana işlem fonksiyonu"""
    start_time = time.time()
    
    # Eşzamanlı istek sayısını sınırla (sitelere saygı için)
    semaphore = asyncio.Semaphore(10)
    
    # Tüm sayfaları asenkron olarak işle
    async with aiohttp.ClientSession() as session:
        tasks = []
        for page_num in range(1, 1000):
            task = process_single_page(session, page_num, semaphore)
            tasks.append(task)
        
        # Tüm görevleri paralel çalıştır
        results = await asyncio.gather(*tasks)
    
    # Tüm linkleri birleştir
    all_links = []
    for links in results:
        all_links.extend(links)
    
    # Benzersiz linkler
    unique_links = list(set(all_links))
    
    # DataFrame oluştur ve kaydet
    df = pd.DataFrame({
        'link': ['/tr' + z.replace('//', '/') for z in unique_links]
    })
    
    df.to_excel('yayin.xlsx', index=False)
    
    end_time = time.time()
    print(f"\nToplam süre: {end_time - start_time:.2f} saniye")
    print(f"Toplam {len(unique_links)} benzersiz yayın linki bulundu")

# Batch processing için alternatif versiyon
async def process_batch(session, page_range, semaphore):
    """Sayfa gruplarını işle"""
    tasks = []
    for page_num in page_range:
        task = process_single_page(session, page_num, semaphore)
        tasks.append(task)
    
    batch_results = await asyncio.gather(*tasks)
    return batch_results

async def main_batched(batch_size=50):
    """Batch processing ile daha kontrollü versiyon"""
    start_time = time.time()
    
    semaphore = asyncio.Semaphore(15)  # Daha fazla eşzamanlı istek
    all_links = []
    
    async with aiohttp.ClientSession() as session:
        for batch_start in range(1, 1000, batch_size):
            batch_end = min(batch_start + batch_size, 1000)
            batch_range = range(batch_start, batch_end)
            
            print(f"Processing batch {batch_start}-{batch_end-1}")
            
            batch_results = await process_batch(session, batch_range, semaphore)
            
            for links in batch_results:
                all_links.extend(links)
            
            # Küçük bir bekleme süresi
            await asyncio.sleep(1)
    
    # Benzersiz linkler
    unique_links = list(set(all_links))
    
    # Kaydet
    df = pd.DataFrame({
        'link': ['/tr' + z.replace('//', '/') for z in unique_links]
    })
    
    df.to_excel('yayin.xlsx', index=False)
    
    end_time = time.time()
    print(f"\nToplam süre: {end_time - start_time:.2f} saniye")
    print(f"Toplam {len(unique_links)} benzersiz yayin linki bulundu")

# Çalıştırma seçenekleri
if __name__ == "__main__":
    # Hızlı versiyon
    print("Hızlı versiyon çalışıyor...")
    asyncio.run(main())
    
    # Veya batch versiyon
    # print("Batch versiyon çalışıyor...")
    # asyncio.run(main_batched())

Hızlı versiyon çalışıyor...
Page 999: Found 0 linkss
Toplam süre: 20.58 saniye
Toplam 1662 benzersiz yayın linki bulundu


In [ ]:
########## Ekibimiz ###########

# Asenkron istekler için gerekli
nest_asyncio.apply()

async def fetch_page(session, url, semaphore, page_id):
    """Tek bir sayfayı asenkron olarak getir"""
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as response:
                if response.status == 200:
                    html = await response.text()
                    return html, page_id, True
                else:
                    print(f"HTTP {response.status} for {url}")
                    return None, page_id, False
        except Exception as e:
            print(f"Error fetching {url}: {str(e)}")
            return None, page_id, False

def is_valid_page(html):
    """Sayfanın geçerli (boş olmayan) içeriğe sahip olup olmadığını kontrol et"""
    if not html:
        return False
    
    soup = BeautifulSoup(html, 'html.parser')
    
    # Geçerli sayfa göstergeleri - BOŞ OLMAYAN sayfaları tespit et
    valid_indicators = [
        # Yazar ismi var mı?
        soup.find('span', class_='bg-black text-white py-2 px-3'),
        # Yazar pozisyonu var mı?
        soup.find('p', class_='text-[13px] font-semibold'),
        # Yazar resmi var mı?
        soup.find('img', alt='Author Image'),
        # İçerik bölümü var mı?
        soup.find('div', class_='text-justify prose prose-sm prose-a:text-blue-600')
    ]
    
    # Bu göstergelerden herhangi biri varsa sayfa geçerlidir
    return any(indicator for indicator in valid_indicators if indicator)

def parse_page_content(html, page_id):
    """Sayfa içeriğini parse et"""
    if not html or not is_valid_page(html):
        return None
    
    soup = BeautifulSoup(html, 'html.parser')
    
    # Sayfa bilgilerini çıkar
    page_data = {
        'id': page_id,
        'url': f"https://www.tepav.org.tr/tr/ekibimiz/s/{page_id}",
        'timestamp': time.time()
    }
    
    # Yazar ismi
    author_name_span = soup.find('span', class_='bg-black text-white py-2 px-3')
    if author_name_span and author_name_span.text.strip():
        page_data['author_name'] = author_name_span.text.strip()
    
    # Yazar pozisyonu
    author_position = soup.find('p', class_='text-[13px] font-semibold')
    if author_position and author_position.text.strip():
        page_data['author_position'] = author_position.text.strip()
    
    # Yazar resmi
    author_img = soup.find('img', alt='Author Image')
    if author_img:
        page_data['author_image'] = author_img.get('src', '')
    
    # Email
    email_link = soup.find('a', href=lambda x: x and 'mailto:' in x)
    if email_link:
        page_data['email'] = email_link.get('href', '').replace('mailto:', '')
    
    # İçerik
    content_div = soup.find('div', class_='text-justify prose prose-sm prose-a:text-blue-600')
    if content_div:
        page_data['content_preview'] = content_div.get_text(strip=True)[:200] + "..." if len(content_div.get_text(strip=True)) > 200 else content_div.get_text(strip=True)
    
    # İlgili yayınlar
    publications = []
    pub_links = soup.find_all('a', href=lambda x: x and '/yayin/s/' in x)
    for link in pub_links[:5]:  # İlk 5 yayın
        publications.append({
            'title': link.get_text(strip=True),
            'url': link.get('href', '')
        })
    if publications:
        page_data['publications'] = publications
    
    # İlgili haberler
    news = []
    news_links = soup.find_all('a', href=lambda x: x and '/haberler/s/' in x)
    for link in news_links[:5]:  # İlk 5 haber
        news.append({
            'title': link.get_text(strip=True),
            'url': link.get('href', '')
        })
    if news:
        page_data['news'] = news
    
    return page_data

async def process_single_id(session, page_id, semaphore):
    """Tek bir ID'yi işle"""
    url = f"https://www.tepav.org.tr/tr/ekibimiz/s/{page_id}"
    
    html, returned_id, success = await fetch_page(session, url, semaphore, page_id)
    
    if success and html:
        page_data = parse_page_content(html, page_id)
        if page_data:
            print(f"✓ Found valid page at ID: {page_id} - {page_data.get('author_name', 'No name')}")
            return page_data
        else:
            print(f"○ Empty at ID: {page_id}", end='\r')
    else:
        print(f"✗ Failed at ID: {page_id}", end='\r')
    
    return None

async def main_full_scan(start_id=1, end_id=2000, batch_size=50):
    """Tam tarama yap"""
    start_time = time.time()
    
    semaphore = asyncio.Semaphore(20)  # Eşzamanlı istek sayısı
    all_valid_pages = []
    total_processed = 0
    found_count = 0
    
    print(f"Starting full scan from ID {start_id} to {end_id}")
    
    async with aiohttp.ClientSession() as session:
        for batch_start in range(start_id, end_id + 1, batch_size):
            batch_end = min(batch_start + batch_size, end_id + 1)
            batch_range = range(batch_start, batch_end)
            
            print(f"\nProcessing batch {batch_start}-{batch_end-1}")
            
            # Batch içindeki tüm ID'leri paralel işle
            tasks = []
            for page_id in batch_range:
                task = process_single_id(session, page_id, semaphore)
                tasks.append(task)
            
            batch_results = await asyncio.gather(*tasks)
            
            # Geçerli sonuçları topla
            batch_valid = [result for result in batch_results if result is not None]
            all_valid_pages.extend(batch_valid)
            found_count += len(batch_valid)
            
            total_processed += len(batch_range)
            progress_percent = (total_processed / (end_id - start_id + 1)) * 100
            print(f"Progress: {total_processed}/{end_id} ({progress_percent:.1f}%) - Found: {found_count} valid pages")
            
            # Küçük bir bekleme süresi (rate limiting)
            await asyncio.sleep(0.3)
    
    # Sonuçları kaydet
    if all_valid_pages:
        # DataFrame oluştur
        df_data = []
        for page in all_valid_pages:
            row = {
                'id': page['id'],
                'url': page['url'],
                'author_name': page.get('author_name', ''),
                'author_position': page.get('author_position', ''),
                'email': page.get('email', ''),
                'author_image': page.get('author_image', ''),
                'content_preview': page.get('content_preview', ''),
                'publications_count': len(page.get('publications', [])),
                'news_count': len(page.get('news', []))
            }
            df_data.append(row)
        
        df = pd.DataFrame(df_data)
        
        # ID'ye göre sırala
        df = df.sort_values('id').reset_index(drop=True)
        
        # Excel'e kaydet
        excel_filename = f'ekibimiz.xlsx'
        df.to_excel(excel_filename, index=False)
        
        end_time = time.time()
        total_time = end_time - start_time
        
        print(f"\n" + "="*50)
        print(f"✓ SCAN COMPLETED SUCCESSFULLY!")
        print(f"✓ Total time: {total_time:.2f} seconds")
        print(f"✓ Scanned range: {start_id}-{end_id}")
        print(f"✓ Total pages scanned: {end_id - start_id + 1}")
        print(f"✓ Valid pages found: {len(all_valid_pages)}")
        print(f"✓ Success rate: {len(all_valid_pages)/(end_id - start_id + 1)*100:.2f}%")
        print(f"✓ Results saved to:")
        print(f"    - {excel_filename}")

        print("="*50)
        
        # İlk birkaç sonucu göster
        if len(all_valid_pages) > 0:
            print("\nFirst few results:")
            for i, page in enumerate(all_valid_pages[:5]):
                print(f"  {i+1}. ID {page['id']}: {page.get('author_name', 'No name')} - {page.get('author_position', 'No position')}")
        
    else:
        print(f"\n✗ No valid pages found in range {start_id}-{end_id}")

# Hızlı test için
async def quick_test():
    """Hızlı test - bilinen bir ID'yi kontrol et"""
    test_id = 27  # Bildiğimiz çalışan ID
    
    semaphore = asyncio.Semaphore(1)
    
    async with aiohttp.ClientSession() as session:
        result = await process_single_id(session, test_id, semaphore)
        
        if result:
            print(f"\n✓ TEST SUCCESSFUL: ID {test_id} is valid")
            print(f"  Author: {result.get('author_name')}")
            print(f"  Position: {result.get('author_position')}")
            return True
        else:
            print(f"\n✗ TEST FAILED: ID {test_id} is not valid")
            return False

# Çalıştırma
if __name__ == "__main__":
    print("TEPAV Ekibimiz Page Scanner")
    print("=" * 40)
    
    # Hızlı test yap
    print("Running quick test...")
    test_result = asyncio.run(quick_test())
    
    if test_result:
        print("\nStarting full scan from ID 1 to 2000...")
        asyncio.run(main_full_scan(start_id=1, end_id=2000, batch_size=50))
    else:
        print("\nTest failed. Please check the website structure or network connection.")
    
ekibimiz = pd.read_excel('ekibimiz.xlsx')
ekibimiz = ekibimiz[['url','author_name']].dropna()
ekibimiz['url'] = [z.replace('https://www.tepav.org.tr','') for z in ekibimiz['url']]
ekibimiz.to_excel('./ekibimiz.xlsx', index=False)

In [ ]:
########## Videolar ###########

# Asenkron istekler için gerekli
nest_asyncio.apply()

async def fetch_page(session, url, semaphore, page_num):
    """Tek bir sayfayı asenkron olarak getir"""
    async with semaphore:
        try:
            async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as response:
                if response.status == 200:
                    html = await response.text()
                    return html, page_num, True
                else:
                    print(f"HTTP {response.status} for {url}")
                    return None, page_num, False
        except Exception as e:
            print(f"Error fetching {url}: {str(e)}")
            return None, page_num, False

def extract_video_details(html, page_num):
    """Video sayfasından detaylı bilgileri çıkar"""
    if not html:
        return []
    
    soup = BeautifulSoup(html, 'html.parser')
    video_data = []
    
    # Tüm linkleri kontrol et
    all_links = soup.find_all(['a', 'iframe'], href=True) + soup.find_all('iframe', src=True)
    
    for element in all_links:
        video_info = {}
        url = element.get('href') or element.get('src', '')
        
        # YouTube embed iframe
        youtube_embed_match = re.search(r'youtube\.com/embed/([a-zA-Z0-9_-]+)', url)
        if youtube_embed_match:
            video_id = youtube_embed_match.group(1)
            video_info = {
                'video_id': video_id,
                'youtube_url': f"https://www.youtube.com/watch?v={video_id}",
                'embed_url': url,
                'page_num': page_num,
                'type': 'embed'
            }
        
        # YouTube watch link
        youtube_watch_match = re.search(r'youtube\.com/watch\?v=([a-zA-Z0-9_-]+)', url)
        if youtube_watch_match and not video_info:
            video_id = youtube_watch_match.group(1)
            video_info = {
                'video_id': video_id,
                'youtube_url': url if url.startswith('http') else f"https://www.youtube.com/watch?v={video_id}",
                'embed_url': f"https://www.youtube.com/embed/{video_id}",
                'page_num': page_num,
                'type': 'watch_link'
            }
        
        # Video başlığını bul
        if video_info:
            parent = element.find_parent()
            for _ in range(3):
                if parent:
                    title_elem = parent.find(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
                    if title_elem and title_elem.text.strip():
                        video_info['title'] = title_elem.text.strip()
                        break
                    parent = parent.find_parent()
            
            if 'title' not in video_info:
                video_info['title'] = f"Video {video_id}"
            
            video_data.append(video_info)
    
    # Benzersiz video ID'leri için filtrele
    unique_videos = {}
    for video in video_data:
        if video['video_id'] not in unique_videos:
            unique_videos[video['video_id']] = video
    
    return list(unique_videos.values())

async def process_single_page(session, page_num, semaphore):
    """Tek bir sayfayı işle"""
    if page_num == 1:
        url = "https://tepav.org.tr/tr/video"
    else:
        url = f"https://tepav.org.tr/tr/video?page={page_num}"
    
    html, returned_num, success = await fetch_page(session, url, semaphore, page_num)
    
    if success and html:
        video_links = extract_video_details(html, page_num)
        if video_links:
            print(f"✓ Page {page_num}: Found {len(video_links)} videos")
            return video_links
        else:
            print(f"○ Page {page_num}: No videos found")
    else:
        print(f"✗ Page {page_num}: Failed to fetch")
    
    return []

async def main_video_scan():
    """Video sayfalarını tarama"""
    start_time = time.time()
    
    semaphore = asyncio.Semaphore(15)
    all_videos = []
    
    print("Starting TEPAV video scan...")
    print("Scanning pages 1 to 50...")
    
    async with aiohttp.ClientSession() as session:
        for batch_start in range(1, 51, 10):
            batch_end = min(batch_start + 10, 51)
            batch_range = range(batch_start, batch_end)
            
            print(f"\nProcessing pages {batch_start}-{batch_end-1}")
            
            tasks = []
            for page_num in batch_range:
                task = process_single_page(session, page_num, semaphore)
                tasks.append(task)
            
            batch_results = await asyncio.gather(*tasks)
            
            batch_videos = []
            for videos in batch_results:
                batch_videos.extend(videos)
            
            all_videos.extend(batch_videos)
            
            print(f"Progress: {batch_end-1}/50 - Total found: {len(all_videos)} videos")
            
            await asyncio.sleep(0.5)
    
    # Benzersiz videoları filtrele
    unique_videos = {}
    for video in all_videos:
        if video['video_id'] not in unique_videos:
            unique_videos[video['video_id']] = video
    
    unique_videos_list = list(unique_videos.values())
    
    # Sonuçları kaydet
    if unique_videos_list:
        df_data = []
        for video in unique_videos_list:
            row = {
                'video_id': video['video_id'],
                'title': video.get('title', ''),
                'youtube_url': video['youtube_url'],
                'embed_url': video.get('embed_url', ''),
                'page_found': video['page_num'],
                'type': video.get('type', 'unknown')
            }
            df_data.append(row)
        
        df = pd.DataFrame(df_data)
        df = df.sort_values('page_found').reset_index(drop=True)
        
        # Excel'e kaydet
        df.to_excel('video.xlsx', index=False)
        
        end_time = time.time()
        total_time = end_time - start_time
        
        print(f"\n" + "="*50)
        print(f"✓ SCAN COMPLETED!")
        print(f"✓ Time: {total_time:.2f} seconds")
        print(f"✓ Pages scanned: 1-50")
        print(f"✓ Total videos found: {len(unique_videos_list)}")
        print(f"✓ File saved: video.xlsx")
        print("="*50)
        
    else:
        print("\n✗ No YouTube videos found")

# Çalıştırma
if __name__ == "__main__":
    asyncio.run(main_video_scan())

video = pd.read_excel('video.xlsx')
video = video[['youtube_url']]
video.columns = ['link']
video.to_excel('./video.xlsx')

In [ ]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse
import nest_asyncio
from collections import deque
import logging

# Asenkron istekler için gerekli
nest_asyncio.apply()

# Logging ayarı
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class TEPAVUltraCrawler:
    def __init__(self):
        self.base_url = "https://www.tepav.org.tr/tr"
        self.base_domain = "tepav.org.tr"
        self.visited_urls = set()
        self.all_internal_links = set()
        self.session = None
        self.semaphore = asyncio.Semaphore(50)  # Çok daha fazla eşzamanlı istek
        self.request_count = 0
        self.start_time = None
    
    def is_internal_link(self, url):
        """URL'nin internal olup olmadığını kontrol et"""
        parsed_url = urlparse(url)
        return parsed_url.netloc.endswith(self.base_domain)
    
    def normalize_url(self, url, base_url):
        """URL'yi normalize et"""
        if not url or url.startswith(('javascript:', 'mailto:', 'tel:', '#')):
            return None
        
        # Tam URL yap
        if url.startswith('http'):
            full_url = url
        else:
            full_url = urljoin(base_url, url)
        
        # Fragment'leri kaldır
        full_url = full_url.split('#')[0]
        
        return full_url
    
    async def fetch_page(self, url):
        """Sayfayı asenkron olarak getir - timeout ve retry mekanizması ile"""
        async with self.semaphore:
            try:
                async with self.session.get(url, timeout=aiohttp.ClientTimeout(total=15)) as response:
                    self.request_count += 1
                    if response.status == 200:
                        html = await response.text()
                        return html, True
                    else:
                        return None, False
            except Exception as e:
                return None, False
    
    def extract_internal_links(self, html, current_url):
        """HTML'den sadece internal linkleri çıkar"""
        if not html:
            return []
        
        soup = BeautifulSoup(html, 'html.parser')
        links = []
        
        # Tüm <a> tag'lerini bul
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            full_url = self.normalize_url(href, current_url)
            
            if full_url and self.is_internal_link(full_url):
                # Aynı sayfaya linkleri ve ziyaret edilenleri engelle
                if (full_url != current_url and 
                    full_url not in self.visited_urls and
                    full_url not in self.all_internal_links):
                    
                    link_info = {
                        'url': full_url,
                        'text': a_tag.get_text(strip=True)[:100],
                        'source_page': current_url,
                        'depth': self.get_url_depth(full_url)
                    }
                    links.append(link_info)
                    self.all_internal_links.add(full_url)
        
        return links
    
    def get_url_depth(self, url):
        """URL'nin derinliğini hesapla"""
        parsed_url = urlparse(url)
        path = parsed_url.path.strip('/')
        if not path:
            return 0
        return len(path.split('/'))
    
    async def process_batch(self, batch_urls, current_depth):
        """URL batch'ini paralel işle"""
        tasks = []
        for url in batch_urls:
            if url not in self.visited_urls:
                task = self.process_single_url(url, current_depth)
                tasks.append(task)
        
        # Tüm batch'i paralel işle
        batch_results = await asyncio.gather(*tasks, return_exceptions=True)
        
        # Tüm yeni linkleri topla
        all_new_links = []
        for result in batch_results:
            if isinstance(result, list):
                all_new_links.extend(result)
        
        return all_new_links
    
    async def process_single_url(self, url, current_depth):
        """Tek bir URL'yi işle"""
        if url in self.visited_urls:
            return []
        
        self.visited_urls.add(url)
        
        # İlerleme göstergesi
        if self.request_count % 100 == 0:
            elapsed = time.time() - self.start_time
            logger.info(f"Progress: {self.request_count} requests, {len(self.visited_urls)} pages, {len(self.all_internal_links)} links, {elapsed:.1f}s")
        
        html, success = await self.fetch_page(url)
        
        if success and html:
            new_links = self.extract_internal_links(html, url)
            return new_links
        
        return []
    
    async def crawl_ultra_deep(self, max_depth=1000, batch_size=100):
        """Ultra derin ve hızlı tarama"""
        self.start_time = time.time()
        
        async with aiohttp.ClientSession() as self.session:
            # Başlangıç URL'si
            current_level_urls = [self.base_url]
            all_discovered_links = []
            current_depth = 0
            
            while current_level_urls and current_depth < max_depth:
                logger.info(f"🔍 Depth {current_depth}: Processing {len(current_level_urls)} URLs...")
                
                # Batch'leri böl
                batches = [current_level_urls[i:i + batch_size] 
                          for i in range(0, len(current_level_urls), batch_size)]
                
                next_level_urls = []
                
                for batch in batches:
                    # Batch'i paralel işle
                    batch_new_links = await self.process_batch(batch, current_depth)
                    all_discovered_links.extend(batch_new_links)
                    
                    # Yeni URL'leri topla
                    for link in batch_new_links:
                        next_level_urls.append(link['url'])
                
                # Benzersiz URL'ler için
                current_level_urls = list(set(next_level_urls))
                current_depth += 1
                
                # Çok fazla URL varsa sınırla
                if len(current_level_urls) > 10000:
                    current_level_urls = current_level_urls[:10000]
                    logger.warning(f"URL sayısı çok fazla, {len(current_level_urls)} ile sınırlandı")
                
                # Derinlik raporu
                logger.info(f"✅ Depth {current_depth-1} completed: {len(batch_new_links)} new links, Next level: {len(current_level_urls)} URLs")
            
            end_time = time.time()
            self.save_results(all_discovered_links, end_time - self.start_time, current_depth)
    
    def save_results(self, links, total_time, max_depth_reached):
        """Sonuçları kaydet"""
        if not links:
            logger.error("No internal links found!")
            return
        
        # DataFrame oluştur
        df_data = []
        for link in links:
            df_data.append({
                'URL': link['url'],
                'Link Text': link['text'],
                'Source Page': link['source_page'],
                'Depth': link['depth']
            })
        
        # Benzersiz URL'ler
        df = pd.DataFrame(df_data)
        df = df.drop_duplicates(subset=['URL']).sort_values(['Depth', 'URL']).reset_index(drop=True)
        
        # Excel'e kaydet
        filename = f'tepav_ultra_deep_links.xlsx'
        df.to_excel(filename, index=False)
        
        # CSV olarak da kaydet (büyük dosyalar için)
        csv_filename = f'tepav_ultra_deep_links.csv'
        df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        
        logger.info(f"\n" + "="*70)
        logger.info(f"🎉 ULTRA DEEP CRAWLING COMPLETED!")
        logger.info(f"⏱️  Total time: {total_time:.2f} seconds")
        logger.info(f"📊 Maximum depth reached: {max_depth_reached}")
        logger.info(f"🔗 Total requests: {self.request_count}")
        logger.info(f"📄 Pages visited: {len(self.visited_urls)}")
        logger.info(f"🔍 Unique internal links found: {len(df)}")
        logger.info(f"💾 Files saved:")
        logger.info(f"   - {filename}")
        logger.info(f"   - {csv_filename}")
        logger.info("="*70)
        
        # Detaylı istatistikler
        depth_stats = df['Depth'].value_counts().sort_index()
        logger.info(f"\n📈 Depth Statistics:")
        for depth, count in depth_stats.items():
            logger.info(f"   Depth {depth}: {count} links")
        
        # Performans istatistikleri
        requests_per_second = self.request_count / total_time if total_time > 0 else 0
        logger.info(f"\n⚡ Performance:")
        logger.info(f"   Requests per second: {requests_per_second:.2f}")
        logger.info(f"   Average time per request: {total_time/self.request_count*1000:.2f}ms")

# Ana fonksiyon
async def main():
    logger.info("🚀 TEPAV Ultra Deep Crawler - 1000 Levels - Starting...")
    logger.info("⚡ Optimized for maximum speed and depth")
    
    crawler = TEPAVUltraCrawler()
    await crawler.crawl_ultra_deep(max_depth=1000, batch_size=200)

# Çalıştırma
if __name__ == "__main__":
    asyncio.run(main())

In [90]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin, urlparse
import nest_asyncio
from collections import deque
import logging

# Asenkron istekler için gerekli
nest_asyncio.apply()

# Logging ayarı
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class TEPAVUltraCrawler:
    def __init__(self):
        self.base_url = "https://www.tepav.org.tr/tr"
        self.base_domain = "tepav.org.tr"
        self.visited_urls = set()
        self.all_internal_links = set()
        self.all_external_links = set()  # Yeni eklenen external linkler için set
        self.session = None
        self.semaphore = asyncio.Semaphore(50)  # Çok daha fazla eşzamanlı istek
        self.request_count = 0
        self.start_time = None
    
    def is_internal_link(self, url):
        """URL'nin internal olup olmadığını kontrol et"""
        parsed_url = urlparse(url)
        return parsed_url.netloc.endswith(self.base_domain)
    
    def normalize_url(self, url, base_url):
        """URL'yi normalize et"""
        if not url or url.startswith(('javascript:', 'mailto:', 'tel:', '#')):
            return None
        
        # Tam URL yap
        if url.startswith('http'):
            full_url = url
        else:
            full_url = urljoin(base_url, url)
        
        # Fragment'leri kaldır
        full_url = full_url.split('#')[0]
        
        return full_url
    
    async def fetch_page(self, url):
        """Sayfayı asenkron olarak getir - timeout ve retry mekanizması ile"""
        async with self.semaphore:
            try:
                async with self.session.get(url, timeout=aiohttp.ClientTimeout(total=15)) as response:
                    self.request_count += 1
                    if response.status == 200:
                        html = await response.text()
                        return html, True
                    else:
                        return None, False
            except Exception as e:
                return None, False
    
    def extract_internal_links(self, html, current_url):
        """HTML'den sadece internal linkleri çıkar"""
        if not html:
            return []
        
        soup = BeautifulSoup(html, 'html.parser')
        links = []
        
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            full_url = self.normalize_url(href, current_url)
            
            if full_url and self.is_internal_link(full_url):
                if (full_url != current_url and 
                    full_url not in self.visited_urls and
                    full_url not in self.all_internal_links):
                    
                    link_info = {
                        'url': full_url,
                        'text': a_tag.get_text(strip=True)[:100],
                        'source_page': current_url,
                        'depth': self.get_url_depth(full_url),
                        'type': 'internal'
                    }
                    links.append(link_info)
                    self.all_internal_links.add(full_url)
        
        return links
    
    def extract_external_links(self, html, current_url):
        """HTML'den external linkleri çıkar"""
        if not html:
            return []
        
        soup = BeautifulSoup(html, 'html.parser')
        links = []
        
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            full_url = self.normalize_url(href, current_url)
            
            if full_url and not self.is_internal_link(full_url) and full_url not in self.all_external_links:
                link_info = {
                    'url': full_url,
                    'text': a_tag.get_text(strip=True)[:100],
                    'source_page': current_url,
                    'depth': self.get_url_depth(current_url),  # External link derinliği source page'e bağlı
                    'type': 'external'
                }
                links.append(link_info)
                self.all_external_links.add(full_url)
        
        return links
    
    def get_url_depth(self, url):
        """URL'nin derinliğini hesapla"""
        parsed_url = urlparse(url)
        path = parsed_url.path.strip('/')
        if not path:
            return 0
        return len(path.split('/'))
    
    async def process_batch(self, batch_urls, current_depth):
        """URL batch'ini paralel işle"""
        tasks = []
        for url in batch_urls:
            if url not in self.visited_urls:
                task = self.process_single_url(url, current_depth)
                tasks.append(task)
        
        batch_results = await asyncio.gather(*tasks, return_exceptions=True)
        
        all_new_links = []
        for result in batch_results:
            if isinstance(result, list):
                all_new_links.extend(result)
        
        return all_new_links
    
    async def process_single_url(self, url, current_depth):
        """Tek bir URL'yi işle"""
        if url in self.visited_urls:
            return []
        
        self.visited_urls.add(url)
        
        if self.request_count % 100 == 0:
            elapsed = time.time() - self.start_time
            logger.info(f"Progress: {self.request_count} requests, {len(self.visited_urls)} pages, {len(self.all_internal_links)} internal links, {len(self.all_external_links)} external links, {elapsed:.1f}s")
        
        html, success = await self.fetch_page(url)
        
        if success and html:
            new_internal_links = self.extract_internal_links(html, url)
            new_external_links = self.extract_external_links(html, url)
            return new_internal_links + new_external_links
        
        return []
    
    async def crawl_ultra_deep(self, max_depth=1000, batch_size=100):
        """Ultra derin ve hızlı tarama"""
        self.start_time = time.time()
        
        async with aiohttp.ClientSession() as self.session:
            current_level_urls = [self.base_url]
            all_discovered_links = []
            current_depth = 0
            
            while current_level_urls and current_depth < max_depth:
                logger.info(f"🔍 Depth {current_depth}: Processing {len(current_level_urls)} URLs...")
                
                batches = [current_level_urls[i:i + batch_size] 
                          for i in range(0, len(current_level_urls), batch_size)]
                
                next_level_urls = []
                
                for batch in batches:
                    batch_new_links = await self.process_batch(batch, current_depth)
                    all_discovered_links.extend(batch_new_links)
                    
                    for link in batch_new_links:
                        if link['type'] == 'internal':
                            next_level_urls.append(link['url'])
                
                current_level_urls = list(set(next_level_urls))
                current_depth += 1
                
                if len(current_level_urls) > 10000:
                    current_level_urls = current_level_urls[:10000]
                    logger.warning(f"URL sayısı çok fazla, {len(current_level_urls)} ile sınırlandı")
                
                logger.info(f"✅ Depth {current_depth-1} completed: {len(batch_new_links)} new links, Next level: {len(current_level_urls)} URLs")
            
            end_time = time.time()
            self.save_results(all_discovered_links, end_time - self.start_time, current_depth)
    
    def save_results(self, links, total_time, max_depth_reached):
        """Sonuçları kaydet"""
        if not links:
            logger.error("No internal or external links found!")
            return
        
        df_data = []
        for link in links:
            df_data.append({
                'URL': link['url'],
                'Link Text': link['text'],
                'Source Page': link['source_page'],
                'Depth': link['depth'],
                'Type': link['type']
            })
        
        df = pd.DataFrame(df_data)
        df = df.drop_duplicates(subset=['URL']).sort_values(['Depth', 'URL']).reset_index(drop=True)
        
        filename = f'tepav_ultra_deep_links.xlsx'
        df.to_excel(filename, index=False)
        
        csv_filename = f'tepav_ultra_deep_links.csv'
        df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
        
        logger.info(f"\n" + "="*70)
        logger.info(f"🎉 ULTRA DEEP CRAWLING COMPLETED!")
        logger.info(f"⏱️  Total time: {total_time:.2f} seconds")
        logger.info(f"📊 Maximum depth reached: {max_depth_reached}")
        logger.info(f"🔗 Total requests: {self.request_count}")
        logger.info(f"📄 Pages visited: {len(self.visited_urls)}")
        logger.info(f"🔍 Unique internal links found: {len(self.all_internal_links)}")
        logger.info(f"🌐 Unique external links found: {len(self.all_external_links)}")
        logger.info(f"💾 Files saved:")
        logger.info(f"   - {filename}")
        logger.info(f"   - {csv_filename}")
        logger.info("="*70)
        
        depth_stats = df['Depth'].value_counts().sort_index()
        logger.info(f"\n📈 Depth Statistics:")
        for depth, count in depth_stats.items():
            logger.info(f"   Depth {depth}: {count} links")
        
        requests_per_second = self.request_count / total_time if total_time > 0 else 0
        logger.info(f"\n⚡ Performance:")
        logger.info(f"   Requests per second: {requests_per_second:.2f}")
        logger.info(f"   Average time per request: {total_time/self.request_count*1000:.2f}ms")

# Ana fonksiyon
async def main():
    logger.info("🚀 TEPAV Ultra Deep Crawler - 1000 Levels - Starting...")
    logger.info("⚡ Optimized for maximum speed and depth")
    
    crawler = TEPAVUltraCrawler()
    await crawler.crawl_ultra_deep(max_depth=1000, batch_size=200)

# Çalıştırma
if __name__ == "__main__":
    asyncio.run(main())

2025-10-02 03:17:39,194 - INFO - 🚀 TEPAV Ultra Deep Crawler - 1000 Levels - Starting...
2025-10-02 03:17:39,195 - INFO - ⚡ Optimized for maximum speed and depth
2025-10-02 03:17:39,195 - INFO - 🔍 Depth 0: Processing 1 URLs...
2025-10-02 03:17:39,196 - INFO - Progress: 0 requests, 1 pages, 0 internal links, 0 external links, 0.0s
2025-10-02 03:17:40,047 - INFO - ✅ Depth 0 completed: 69 new links, Next level: 60 URLs
2025-10-02 03:17:40,048 - INFO - 🔍 Depth 1: Processing 60 URLs...
2025-10-02 03:17:50,547 - INFO - ✅ Depth 1 completed: 572 new links, Next level: 557 URLs
2025-10-02 03:17:50,547 - INFO - 🔍 Depth 2: Processing 557 URLs...
2025-10-02 03:18:33,903 - INFO - ✅ Depth 2 completed: 614 new links, Next level: 1754 URLs
2025-10-02 03:18:33,904 - INFO - 🔍 Depth 3: Processing 1754 URLs...
2025-10-02 03:20:43,281 - INFO - ✅ Depth 3 completed: 306 new links, Next level: 2591 URLs
2025-10-02 03:20:43,282 - INFO - 🔍 Depth 4: Processing 2591 URLs...
2025-10-02 03:23:46,900 - INFO - ✅ Depth

In [2]:
import pandas as pd
all = pd.read_excel('tepav_ultra_deep_links.xlsx')

In [ ]:
for w in ['proje','arastirmacilar','ekibimiz','calismalarimiz','altbirim','haberler','blog','yayin','video','podcast','kurumsal']:
    b = all[[w in z for z in all['URL']]]

In [ ]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
import time
import nest_asyncio
import logging
from tqdm import tqdm
import datetime

# Asenkron istekler için gerekli
nest_asyncio.apply()

# Logging ayarı - daha az verbose
logging.basicConfig(level=logging.WARNING, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class FastLinkChecker:
    def __init__(self, max_concurrent=50):  # Daha fazla eşzamanlı istek
        self.max_concurrent = max_concurrent
        self.session = None
        self.results = []
        self.start_time = None
        self.completed_count = 0
        self.total_count = 0
        
    async def __aenter__(self):
        # Daha agresif timeout ve connection ayarları
        timeout = aiohttp.ClientTimeout(total=15, connect=5)  # Daha kısa timeout
        connector = aiohttp.TCPConnector(
            limit=self.max_concurrent, 
            limit_per_host=20,  # Aynı host için daha fazla bağlantı
            use_dns_cache=True,
            ttl_dns_cache=300
        )
        self.session = aiohttp.ClientSession(
            timeout=timeout, 
            connector=connector,
            headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
        )
        return self
        
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.session:
            await self.session.close()
    
    def fast_is_404_page(self, html_content):
        """
        Hızlı 404 kontrolü - daha az string işlemi
        """
        # Önce hızlı string kontrolleri
        if 'next-error-h1' in html_content:
            return True
            
        if 'This page could not be found' in html_content:
            return True
            
        # Sadece küçük bir örnekte kontrol yap
        sample_text = html_content[:5000].lower()  # Sadece ilk 5000 karakter
        
        if '404' in sample_text:
            # Hızlı hata indikatörleri
            error_indicators = [
                'page could not be found',
                'sayfa bulunamadı',
                'not found',
                'hata sayfası'
            ]
            
            for indicator in error_indicators:
                if indicator in sample_text:
                    return True
        
        return False
    
    async def check_url(self, url, pbar):
        """
        Hızlı URL kontrolü - daha az işlem
        """
        try:
            # HEAD isteği ile hızlı kontrol (daha hızlı)
            async with self.session.head(url, allow_redirects=True) as head_response:
                status_code = head_response.status
                
                # Eğer 200 değilse veya şüpheli durum varsa detaylı kontrol
                if status_code != 200:
                    async with self.session.get(url, allow_redirects=True) as get_response:
                        html_content = await get_response.text()
                        is_404 = self.fast_is_404_page(html_content)
                else:
                    # 200 ise hızlı içerik kontrolü
                    try:
                        async with self.session.get(url, allow_redirects=True) as get_response:
                            html_content = await get_response.read()  # read() daha hızlı
                            html_content = html_content.decode('utf-8', errors='ignore')
                            is_404 = self.fast_is_404_page(html_content)
                    except:
                        is_404 = False  # İndirme hatası varsa 404 değil kabul et
                
                result = {
                    'url': url,
                    'status_code': status_code,
                    'is_404': is_404,
                    'error': None,
                    'final_url': str(head_response.url)
                }
                
        except asyncio.TimeoutError:
            result = {
                'url': url,
                'status_code': None,
                'is_404': True,
                'error': 'Timeout',
                'final_url': url
            }
        except Exception as e:
            result = {
                'url': url,
                'status_code': None,
                'is_404': True,
                'error': str(e),
                'final_url': url
            }
        
        self.completed_count += 1
        pbar.update(1)
        self.update_progress(pbar)
        
        return result
    
    def update_progress(self, pbar):
        """İlerleme durumunu güncelle"""
        if self.start_time and self.total_count > 0:
            elapsed_time = time.time() - self.start_time
            if elapsed_time > 1:  # En az 1 saniye geçmişse
                items_per_second = self.completed_count / elapsed_time
                remaining_items = self.total_count - self.completed_count
                estimated_remaining = remaining_items / items_per_second
                
                pbar.set_postfix({
                    'Hız': f'{items_per_second:.1f} link/sn',
                    'Kalan': f'{estimated_remaining:.0f}s',
                    'Tamamlanan': f'{self.completed_count}/{self.total_count}'
                })
    
    async def check_urls(self, urls):
        """
        URL listesini hızlı şekilde kontrol et
        """
        self.start_time = time.time()
        self.total_count = len(urls)
        self.completed_count = 0
        
        with tqdm(total=len(urls), desc="🔗 Linkler kontrol ediliyor", 
                 unit="link", ncols=100) as pbar:
            
            # Daha hızlı task oluşturma
            semaphore = asyncio.Semaphore(self.max_concurrent)
            
            async def bounded_check(url):
                async with semaphore:
                    return await self.check_url(url, pbar)
            
            # Tüm task'leri bir kerede oluştur
            tasks = [bounded_check(url) for url in urls]
            results = await asyncio.gather(*tasks, return_exceptions=True)
            
            # Hızlı hata işleme
            processed_results = []
            for result in results:
                if isinstance(result, Exception):
                    processed_results.append({
                        'url': 'unknown',
                        'status_code': None,
                        'is_404': True,
                        'error': str(result),
                        'final_url': 'unknown'
                    })
                else:
                    processed_results.append(result)
            
            self.results = processed_results
            return processed_results

def fast_check_excel_structure():
    """
    Excel dosyasını hızlı şekilde oku
    """
    try:
        # Daha hızlı okuma için engine belirt
        df = pd.read_excel('tepav_ultra_deep_links.xlsx', engine='openpyxl')
        
        print("📊 Excel yapısı:")
        print(f"   Sütunlar: {df.columns.tolist()}")
        print(f"   Toplam satır: {len(df)}")
        
        # Hızlı sütun bulma
        url_column = next((col for col in df.columns if 'url' in col.lower()), None)
        type_column = next((col for col in df.columns if 'type' in col.lower()), None)
        
        print(f"   URL sütunu: {url_column}")
        print(f"   Type sütunu: {type_column}")
        
        return df, url_column, type_column
        
    except Exception as e:
        print(f"❌ Excel okuma hatası: {e}")
        return None, None, None

async def main():
    print("🚀 HIZLI TEPAV Link Kontrolü Başlatılıyor...")
    print("=" * 50)
    
    start_prep = time.time()
    
    # Excel'i hızlı oku
    df, url_column, type_column = fast_check_excel_structure()
    
    if df is None or not url_column or not type_column:
        print("❌ Excel dosyası veya gerekli sütunlar bulunamadı!")
        return
    
    # Internal linkleri hızlı filtrele
    internal_mask = (df[type_column] == 'internal') & (df[url_column].notna())
    urls = df.loc[internal_mask, url_column].tolist()
    
    prep_time = time.time() - start_prep
    
    print(f"📋 Internal link sayısı: {len(urls)}")
    print(f"⚡ Hazırlık süresi: {prep_time:.2f}s")
    
    if not urls:
        print("⚠️ Internal link bulunamadı!")
        return
    
    # Tahmini süre (daha agresif)
    estimated_time = max(len(urls) * 0.1, 5)  # Her link için ~0.1s
    print(f"⏱️  Tahmini süre: {estimated_time:.1f}s")
    print(f"🕒 Başlangıç: {datetime.datetime.now().strftime('%H:%M:%S')}")
    print("=" * 50)
    
    # Hızlı checker ile kontrol
    async with FastLinkChecker(max_concurrent=50) as checker:  # Daha fazla concurrent
        await checker.check_urls(urls)
        
        # Sonuçları hızlı işle
        results_df = pd.DataFrame(checker.results)
        
        # Hızlı merge
        final_df = df[internal_mask].copy()
        for col in ['status_code', 'is_404', 'error', 'final_url']:
            final_df[col] = results_df[col].values
        
        # Hızlı kaydet
        output_file = f'tepav_link_results_{int(time.time())}.xlsx'
        final_df.to_excel(output_file, index=False, engine='openpyxl')
        
        total_time = time.time() - start_prep
        
        # İstatistikler
        total_links = len(final_df)
        broken_links = final_df['is_404'].sum()
        working_links = total_links - broken_links
        
        print("\n" + "=" * 50)
        print("✅ KONTROL TAMAMLANDI!")
        print("=" * 50)
        print(f"📊 Sonuçlar:")
        print(f"   Toplam: {total_links} link")
        print(f"   ✅ Çalışan: {working_links}")
        print(f"   ❌ Bozuk: {broken_links}")
        print(f"   📈 Başarı: {working_links/total_links*100:.1f}%")
        print(f"   ⚡ Toplam süre: {total_time:.1f}s")
        print(f"   🚀 Ortalama hız: {total_links/total_time:.1f} link/sn")
        print(f"   💾 Dosya: {output_file}")
        
        if broken_links > 0:
            print(f"\n🔴 Bozuk linkler ({broken_links}):")
            broken_urls = final_df[final_df['is_404'] == True][url_column].head(10)  # İlk 10'u göster
            for url in broken_urls:
                print(f"   ❌ {url}")

if __name__ == "__main__":
    # Daha hızlı event loop
    asyncio.run(main())

🚀 HIZLI TEPAV Link Kontrolü Başlatılıyor...
📊 Excel yapısı:
   Sütunlar: ['URL', 'Link Text', 'Source Page', 'Depth', 'Type']
   Toplam satır: 56281
   URL sütunu: URL
   Type sütunu: Type
📋 Internal link sayısı: 89
⚡ Hazırlık süresi: 3.17s
⏱️  Tahmini süre: 8.9s
🕒 Başlangıç: 08:44:33


🔗 Linkler kontrol ediliyor: 100%|█| 89/89 [00:05<00:00, 17.37link/s, Hız=17.4 link/sn, Kalan=0s, Ta


ValueError: Length of values (89) does not match length of index (41786)

In [3]:
import asyncio
import aiohttp
import pandas as pd
from bs4 import BeautifulSoup
import time
import nest_asyncio
import logging
from tqdm import tqdm
import datetime

# Asenkron istekler için gerekli
nest_asyncio.apply()

# Logging ayarı - daha az verbose
logging.basicConfig(level=logging.WARNING, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class FastLinkChecker:
    def __init__(self, max_concurrent=50):  # Daha fazla eşzamanlı istek
        self.max_concurrent = max_concurrent
        self.session = None
        self.results = []
        self.start_time = None
        self.completed_count = 0
        self.total_count = 0
        
    async def __aenter__(self):
        # Daha agresif timeout ve connection ayarları
        timeout = aiohttp.ClientTimeout(total=15, connect=5)  # Daha kısa timeout
        connector = aiohttp.TCPConnector(
            limit=self.max_concurrent, 
            limit_per_host=20,  # Aynı host için daha fazla bağlantı
            use_dns_cache=True,
            ttl_dns_cache=300
        )
        self.session = aiohttp.ClientSession(
            timeout=timeout, 
            connector=connector,
            headers={
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }
        )
        return self
        
    async def __aexit__(self, exc_type, exc_val, exc_tb):
        if self.session:
            await self.session.close()
    
    def fast_is_404_page(self, html_content):
        """
        Hızlı 404 kontrolü - daha az string işlemi
        """
        # Önce hızlı string kontrolleri
        if 'next-error-h1' in html_content:
            return True
            
        if 'This page could not be found' in html_content:
            return True
            
        # Sadece küçük bir örnekte kontrol yap
        sample_text = html_content[:5000].lower()  # Sadece ilk 5000 karakter
        
        if '404' in sample_text:
            # Hızlı hata indikatörleri
            error_indicators = [
                'page could not be found',
                'sayfa bulunamadı',
                'not found',
                'hata sayfası'
            ]
            
            for indicator in error_indicators:
                if indicator in sample_text:
                    return True
        
        return False
    
    async def check_url(self, url, pbar):
        """
        Hızlı URL kontrolü - daha az işlem
        """
        try:
            # HEAD isteği ile hızlı kontrol (daha hızlı)
            async with self.session.head(url, allow_redirects=True) as head_response:
                status_code = head_response.status
                
                # Eğer 200 değilse veya şüpheli durum varsa detaylı kontrol
                if status_code != 200:
                    async with self.session.get(url, allow_redirects=True) as get_response:
                        html_content = await get_response.text()
                        is_404 = self.fast_is_404_page(html_content)
                else:
                    # 200 ise hızlı içerik kontrolü
                    try:
                        async with self.session.get(url, allow_redirects=True) as get_response:
                            html_content = await get_response.read()  # read() daha hızlı
                            html_content = html_content.decode('utf-8', errors='ignore')
                            is_404 = self.fast_is_404_page(html_content)
                    except:
                        is_404 = False  # İndirme hatası varsa 404 değil kabul et
                
                result = {
                    'url': url,
                    'status_code': status_code,
                    'is_404': is_404,
                    'error': None,
                    'final_url': str(head_response.url)
                }
                
        except asyncio.TimeoutError:
            result = {
                'url': url,
                'status_code': None,
                'is_404': True,
                'error': 'Timeout',
                'final_url': url
            }
        except Exception as e:
            result = {
                'url': url,
                'status_code': None,
                'is_404': True,
                'error': str(e),
                'final_url': url
            }
        
        self.completed_count += 1
        pbar.update(1)
        self.update_progress(pbar)
        
        return result
    
    def update_progress(self, pbar):
        """İlerleme durumunu güncelle"""
        if self.start_time and self.total_count > 0:
            elapsed_time = time.time() - self.start_time
            if elapsed_time > 1:  # En az 1 saniye geçmişse
                items_per_second = self.completed_count / elapsed_time
                remaining_items = self.total_count - self.completed_count
                estimated_remaining = remaining_items / items_per_second
                
                pbar.set_postfix({
                    'Hız': f'{items_per_second:.1f} link/sn',
                    'Kalan': f'{estimated_remaining:.0f}s',
                    'Tamamlanan': f'{self.completed_count}/{self.total_count}'
                })
    
    async def check_urls(self, urls):
        """
        URL listesini hızlı şekilde kontrol et
        """
        self.start_time = time.time()
        self.total_count = len(urls)
        self.completed_count = 0
        
        with tqdm(total=len(urls), desc="🔗 Linkler kontrol ediliyor", 
                 unit="link", ncols=100) as pbar:
            
            # Daha hızlı task oluşturma
            semaphore = asyncio.Semaphore(self.max_concurrent)
            
            async def bounded_check(url):
                async with semaphore:
                    return await self.check_url(url, pbar)
            
            # Tüm task'leri bir kerede oluştur
            tasks = [bounded_check(url) for url in urls]
            results = await asyncio.gather(*tasks, return_exceptions=True)
            
            # Hızlı hata işleme
            processed_results = []
            for result in results:
                if isinstance(result, Exception):
                    processed_results.append({
                        'url': 'unknown',
                        'status_code': None,
                        'is_404': True,
                        'error': str(result),
                        'final_url': 'unknown'
                    })
                else:
                    processed_results.append(result)
            
            self.results = processed_results
            return processed_results

def fast_check_excel_structure():
    """
    Excel dosyasını hızlı şekilde oku
    """
    try:
        # Daha hızlı okuma için engine belirt
        df = pd.read_excel('tepav_ultra_deep_links.xlsx', engine='openpyxl')
        
        print("📊 Excel yapısı:")
        print(f"   Sütunlar: {df.columns.tolist()}")
        print(f"   Toplam satır: {len(df)}")
        
        # Hızlı sütun bulma
        url_column = next((col for col in df.columns if 'url' in col.lower()), None)
        type_column = next((col for col in df.columns if 'type' in col.lower()), None)
        
        print(f"   URL sütunu: {url_column}")
        print(f"   Type sütunu: {type_column}")
        
        return df, url_column, type_column
        
    except Exception as e:
        print(f"❌ Excel okuma hatası: {e}")
        return None, None, None

async def main():
    print("🚀 HIZLI TEPAV Link Kontrolü Başlatılıyor...")
    print("🎯 MOD: İLK 100 VERİ İLE TEST")
    print("=" * 50)
    
    start_prep = time.time()
    
    # Excel'i hızlı oku
    df, url_column, type_column = fast_check_excel_structure()
    
    if df is None or not url_column or not type_column:
        print("❌ Excel dosyası veya gerekli sütunlar bulunamadı!")
        return
    
    # Internal linkleri hızlı filtrele ve sadece ilk 100'ü al
    internal_mask = (df[type_column] == 'internal') & (df[url_column].notna())
    all_internal_urls = df.loc[internal_mask, url_column].tolist()
    
    # SADECE İLK 100 URL'yi al
    urls = all_internal_urls[:100]
    
    prep_time = time.time() - start_prep
    
    print(f"📋 Toplam internal link: {len(all_internal_urls)}")
    print(f"🎯 Test için alınan: {len(urls)} link (ilk 100)")
    print(f"⚡ Hazırlık süresi: {prep_time:.2f}s")
    
    if not urls:
        print("⚠️ Internal link bulunamadı!")
        return
    
    print(f"\n🔍 Kontrol edilecek ilk 10 URL:")
    for i, url in enumerate(urls[:10]):
        print(f"   {i+1}. {url}")
    
    if len(urls) > 10:
        print(f"   ... ve {len(urls) - 10} link daha")
    
    # Tahmini süre (daha agresif)
    estimated_time = max(len(urls) * 0.1, 3)  # Her link için ~0.1s
    print(f"\n⏱️  Tahmini süre: {estimated_time:.1f}s")
    print(f"🕒 Başlangıç: {datetime.datetime.now().strftime('%H:%M:%S')}")
    print("=" * 50)
    
    # Hızlı checker ile kontrol
    async with FastLinkChecker(max_concurrent=50) as checker:  # Daha fazla concurrent
        await checker.check_urls(urls)
        
        # Sonuçları hızlı işle
        results_df = pd.DataFrame(checker.results)
        
        # Orijinal datadan sadece ilk 100'ü al
        test_df = df[internal_mask].head(100).copy()
        
        # Sonuçları birleştir
        for col in ['status_code', 'is_404', 'error', 'final_url']:
            test_df[col] = results_df[col].values
        
        # Hızlı kaydet
        output_file = f'tepav_TEST_100_link_{int(time.time())}.xlsx'
        test_df.to_excel(output_file, index=False, engine='openpyxl')
        
        total_time = time.time() - start_prep
        
        # İstatistikler
        total_links = len(test_df)
        broken_links = test_df['is_404'].sum()
        working_links = total_links - broken_links
        
        print("\n" + "=" * 50)
        print("✅ TEST TAMAMLANDI!")
        print("=" * 50)
        print(f"📊 Sonuçlar (İlk 100 link):")
        print(f"   Toplam: {total_links} link")
        print(f"   ✅ Çalışan: {working_links}")
        print(f"   ❌ Bozuk: {broken_links}")
        print(f"   📈 Başarı: {working_links/total_links*100:.1f}%")
        print(f"   ⚡ Toplam süre: {total_time:.1f}s")
        print(f"   🚀 Ortalama hız: {total_links/total_time:.1f} link/sn")
        print(f"   💾 Dosya: {output_file}")
        
        if broken_links > 0:
            print(f"\n🔴 Bozuk linkler ({broken_links}):")
            broken_df = test_df[test_df['is_404'] == True]
            for idx, row in broken_df.iterrows():
                status_info = f"HTTP {row['status_code']}" if row['status_code'] else "Hata"
                error_info = f" - {row['error']}" if row['error'] else ""
                print(f"   ❌ {row[url_column]} ({status_info}{error_info})")
        else:
            print(f"\n🎉 İlk 100 linkte hiç bozuk link bulunamadı!")
        
        # Örnek çalışan linkler
        working_df = test_df[test_df['is_404'] == False]
        if len(working_df) > 0:
            print(f"\n✅ Örnek çalışan linkler:")
            for idx, row in working_df.head(3).iterrows():
                print(f"   ✓ {row[url_column]}")

if __name__ == "__main__":
    # Daha hızlı event loop
    asyncio.run(main())

🚀 HIZLI TEPAV Link Kontrolü Başlatılıyor...
🎯 MOD: İLK 100 VERİ İLE TEST
📊 Excel yapısı:
   Sütunlar: ['URL', 'Link Text', 'Source Page', 'Depth', 'Type']
   Toplam satır: 56281
   URL sütunu: URL
   Type sütunu: Type
📋 Toplam internal link: 41786
🎯 Test için alınan: 100 link (ilk 100)
⚡ Hazırlık süresi: 3.13s

🔍 Kontrol edilecek ilk 10 URL:
   1. http://www.tepav.org.tr
   2. http://www.tepav.org.tr/
   3. https://kobiekarne.tepav.org.tr/
   4. https://tepav.org.tr
   5. https://tepav.org.tr/
   6. https://www.tepav.org.tr/
   7. http://tepav.org.tr/eu/
   8. http://tepav.org.tr/reach/
   9. http://www.tepav.org.tr/1MayisEtkiAnaliziToplantiResimleri.rar
   10. http://www.tepav.org.tr/arastirmacilar
   ... ve 90 link daha

⏱️  Tahmini süre: 10.0s
🕒 Başlangıç: 08:47:23


🔗 Linkler kontrol ediliyor: 100%|█| 100/100 [00:06<00:00, 16.52link/s, Hız=16.5 link/sn, Kalan=0s, 



✅ TEST TAMAMLANDI!
📊 Sonuçlar (İlk 100 link):
   Toplam: 100 link
   ✅ Çalışan: 0
   ❌ Bozuk: 100
   📈 Başarı: 0.0%
   ⚡ Toplam süre: 9.2s
   🚀 Ortalama hız: 10.8 link/sn
   💾 Dosya: tepav_TEST_100_link_1759384049.xlsx

🔴 Bozuk linkler (100):
   ❌ http://www.tepav.org.tr (HTTP 200.0)
   ❌ http://www.tepav.org.tr/ (HTTP 200.0)
   ❌ https://kobiekarne.tepav.org.tr/ (HTTP nan - Cannot connect to host kobiekarne.tepav.org.tr:443 ssl:default [getaddrinfo failed])
   ❌ https://tepav.org.tr (HTTP 200.0)
   ❌ https://tepav.org.tr/ (HTTP 200.0)
   ❌ https://www.tepav.org.tr/ (HTTP 200.0)
   ❌ http://tepav.org.tr/eu/ (HTTP 404.0)
   ❌ http://tepav.org.tr/reach/ (HTTP 404.0)
   ❌ http://www.tepav.org.tr/1MayisEtkiAnaliziToplantiResimleri.rar (HTTP 404.0)
   ❌ http://www.tepav.org.tr/arastirmacilar (HTTP 200.0)
   ❌ http://www.tepav.org.tr/bagis (HTTP 200.0)
   ❌ http://www.tepav.org.tr/blog (HTTP 200.0)
   ❌ http://www.tepav.org.tr/ekibimiz (HTTP 200.0)
   ❌ http://www.tepav.org.tr/haberler (HTT